**TRAINING in CrewAI means improving a crew’s future responses through repeated task execution and human feedback.**

In [1]:
import os
from crewai import Agent, Task, Crew, LLM
from dotenv import load_dotenv
import asyncio

load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")

llm = LLM(model="gpt-4o-mini", temperature=0)

topic = input("Enter a topic: ")

teacher = Agent(
    role="AI Teacher",
    goal="Explain concepts in simple language.",
    backstory="You teach beginners using everyday examples.",
    llm=llm,
    allow_delegation=False,
    verbose=False
)

task = Task(
    description="Explain {topic} in two simple sentences.",
    expected_output="A beginner-friendly explanation with one example.",
    agent=teacher
)

crew = Crew(
    agents=[teacher],
    tasks=[task],
    verbose=False
)

await asyncio.to_thread(
    crew.train,
    n_iterations=1,
    inputs={"topic": topic},
    filename="training_feedback.pkl"
)

print("Training completed.")
result = await crew.kickoff_async(inputs={"topic": topic})

print("\nTopic:", topic)
print("Answer:", result.raw)

Enter a topic:  What is Crew AI?


╭───────────────────────────────────────── 🎓 Training Feedback Required ─────────────────────────────────────────╮
│                                                                                                                 │
│  TRAINING MODE: Provide feedback to improve the agent's performance.                                            │
│                                                                                                                 │
│  This will be used to train better versions of the agent.                                                       │
│  Please provide detailed feedback about the result quality and reasoning process.                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Use simple words


Processing your feedback...

Training completed.

Topic: What is Crew AI?
Answer: Crew AI is a type of artificial intelligence designed to help teams work better together by organizing tasks, sharing information, and improving communication. For example, imagine a group project where Crew AI helps everyone keep track of their responsibilities and deadlines, making it easier for the team to collaborate and succeed.


In [2]:
import pickle
from pprint import pprint

with open("training_feedback.pkl", "rb") as file:
    training_data = pickle.load(file)

pprint(training_data)

{'AI Teacher': {'final_summary': '1. Review the initial output for complex '
                                 'vocabulary and replace it with simpler '
                                 'alternatives. 2. Incorporate relatable '
                                 'examples that resonate with the target '
                                 'audience. 3. Ensure that explanations are '
                                 'clear and concise to facilitate better '
                                 'understanding.',
                'quality': 8.0,
                'suggestions': ['Use simpler vocabulary and avoid complex '
                                'terms to enhance understanding.',
                                'Provide relatable examples that are easy to '
                                'grasp for a wider audience.',
                                'Focus on clarity and conciseness in '
                                'explanations to improve communication '
                                '

**REASONING is the process of thinking through a problem, breaking it into steps, and deciding how to solve it.**

**reasoning=True enables planning and reflection before task execution.**<br>
**max_reasoning_attempts=2 limits attempts to refine that plan; it does not mean two task executions.**

In [ ]:
import os
from crewai import Agent, Task, Crew, LLM
from dotenv import load_dotenv

load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")
llm = LLM(model="gpt-4o-mini", temperature=0)

teacher = Agent(
    role="AI Teacher",
    goal="Explain concepts simply.",
    backstory="You teach beginners using everyday examples.",
    reasoning=True,
    llm=llm,
    max_reasoning_attempts=1,
    verbose=False
)

task = Task(
    description="Explain artificial intelligence in two simple sentences.",
    expected_output="A simple explanation with one everyday example.",
    agent=teacher
)

crew = Crew(
    agents=[teacher],
    tasks=[task],
    verbose=False
)

result = await crew.kickoff_async()
print(result.raw)

**Reasoning with Logs**

In [ ]:
import logging
from crewai import Agent, Task, Crew, LLM

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s",
    force=True
)
logger = logging.getLogger("study_planner")

llm = LLM(model="gpt-4o-mini", temperature=0)

agent = Agent(
    role="Study Planner",
    goal="Create simple study plans for beginners.",
    backstory="You help students learn topics step by step.",
    llm=llm,
    reasoning=True,
    max_reasoning_attempts=3,
    verbose=False
)

task = Task(
    description="Create a 3-day beginner study plan for {topic}.",
    expected_output="One topic and one practice activity per day.",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)

topic = input("Enter a topic: ")

try:
    logger.info("Starting study plan with reasoning enabled.")

    result = await crew.kickoff_async(inputs={"topic": topic})

    logger.info("Study plan completed.")
    print("\nAnswer:\n", result.raw)

except Exception:
    logger.exception("Study plan execution failed.")